In [10]:
#Note - Before running the code delete all csv and pkl files present in this folder except 'housing.csv'
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

MODEL_FILE = 'model.pkl'
PIPELINE_FILE = 'pipeline.pkl'
FILE_NAME = 'housing.csv'
TEST_FILE = 'test.csv'
TEST_OUTPUT_FILE = 'test_output.csv'
PREDICTED_FILE = 'predicted.csv'

def build_pipeline (num_attributes, cat_attributes):
    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scalar', StandardScaler())
    ])
    cat_pipeline = Pipeline([
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])
    full_pipeline = ColumnTransformer([
        ('num', num_pipeline, num_attributes),
        ('cat', cat_pipeline, cat_attributes)
    ])
    return full_pipeline

if not os.path.exists(MODEL_FILE):
    df = pd.read_csv(FILE_NAME)

    df['income_cat'] = pd.cut(df['median_house_value'], bins = [0, 1.5, 3, 4.5, 6, np.inf], labels = [1, 2, 3, 4, 5])

    split = StratifiedShuffleSplit(n_splits = 1, test_size=0.2, random_state = 42)

    for train_index, test_index in split.split(df, df['income_cat']):
        train_data = df.loc[train_index].drop('income_cat', axis = 1)
        test_data = df.loc[test_index].drop('income_cat', axis = 1)

    test_data['median_house_value'].to_csv(TEST_OUTPUT_FILE)
    test_data = test_data.drop('median_house_value', axis = 1).to_csv(TEST_FILE)

    housing = train_data.drop('median_house_value', axis = 1)
    housing_labels = train_data['median_house_value']

    num_attributes = housing.drop('ocean_proximity', axis = 1).columns.tolist()
    cat_attributes = ['ocean_proximity']

    pipeline = build_pipeline(num_attributes, cat_attributes)
    housing_prepared = pipeline.fit_transform(housing)

    model = RandomForestRegressor(random_state = 42)
    model.fit(housing_prepared, housing_labels)

    joblib.dump(model, MODEL_FILE)
    joblib.dump(pipeline, PIPELINE_FILE)

    print('Model is Trained & Saved Successfully')

else:

    model = joblib.load(MODEL_FILE)
    pipeline = joblib.load(PIPELINE_FILE)

    test_output_data = pd.read_csv(TEST_OUTPUT_FILE)
    input_data = pd.read_csv(TEST_FILE)
    columns = input_data.columns.tolist()
    

    transformed_input = pipeline.transform(input_data)
    prediction = model.predict(transformed_input)

    final_output = pd.DataFrame()
    final_output = input_data.copy()
    final_output['median_house_value'] = test_output_data['median_house_value']
    final_output['predicted_median_house_value'] = prediction
    final_output.to_csv(PREDICTED_FILE)
    print("Inference complete. Results saved to output.csv")

    


Inference complete. Results saved to output.csv
